# Sistemas Multiagente

Um agente único acumula no próprio contexto tudo o que leu e divide o mesmo orçamento de buscas e de tokens entre todas as partes da tarefa. Este notebook checa os fatos de um post sobre Natal, primeiro com um agente único e depois com uma equipe de checadores e um redator, e mede o que a divisão ganha e o que ela custa.

In [1]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [2]:
import re
import time

import pandas as pd
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

In [3]:
model = init_chat_model("openai:gpt-4.1-mini", temperature=0.0)

web_search = {"type": "web_search"}

## Agente único

O post abaixo é uma lista de curiosidades sobre Natal com oito afirmações verificáveis, das quais quatro são falsas. O gabarito em `ANSWERS` serve para medir o acerto de cada versão.

In [4]:
CLAIMS = [
    "Natal foi fundada em 25 de dezembro de 1599.",
    "A Ponte Newton Navarro foi inaugurada em 2010.",
    "A Arena das Dunas recebeu seis jogos da Copa do Mundo de 2014.",
    "No Censo de 2022, Natal tinha cerca de 751 mil habitantes.",
    "A UFRN foi criada em 1958.",
    "Luís da Câmara Cascudo nasceu em Natal em 1908.",
    "O Instituto Metrópole Digital da UFRN foi criado em 2011.",
    "Roosevelt e Getúlio Vargas se encontraram em Natal em janeiro de 1944.",
]

ANSWERS = [
    "VERDADEIRO", "FALSO", "FALSO", "VERDADEIRO",
    "VERDADEIRO", "FALSO", "VERDADEIRO", "FALSO",
]

In [5]:
post = "Curiosidades sobre Natal que todo potiguar deveria saber:\n"
for number, claim in enumerate(CLAIMS, start=1):
    post += f"{number}. {claim}\n"

QUESTION = """Verifique as afirmações do post abaixo.
Para cada uma, responda em uma linha que começa com o número e o veredito
(VERDADEIRO ou FALSO), seguida do dado correto e da fonte (URL).

# Post
""" + post

print(QUESTION)

Verifique as afirmações do post abaixo.
Para cada uma, responda em uma linha que começa com o número e o veredito
(VERDADEIRO ou FALSO), seguida do dado correto e da fonte (URL).

# Post
Curiosidades sobre Natal que todo potiguar deveria saber:
1. Natal foi fundada em 25 de dezembro de 1599.
2. A Ponte Newton Navarro foi inaugurada em 2010.
3. A Arena das Dunas recebeu seis jogos da Copa do Mundo de 2014.
4. No Censo de 2022, Natal tinha cerca de 751 mil habitantes.
5. A UFRN foi criada em 1958.
6. Luís da Câmara Cascudo nasceu em Natal em 1908.
7. O Instituto Metrópole Digital da UFRN foi criado em 2011.
8. Roosevelt e Getúlio Vargas se encontraram em Natal em janeiro de 1944.



O agente de referência usa a busca na web da OpenAI como ferramenta do provedor: o dicionário só declara a ferramenta, a busca roda no servidor, e os resultados entram na mesma chamada ao modelo.

In [6]:
SINGLE = """# Papel
Você é um checador de fatos.

# Regras
Pesquise na web antes de responder."""

single = create_agent(model, tools=[web_search], system_prompt=SINGLE)

start = time.perf_counter()
single_result = single.invoke({"messages": [{"role": "user", "content": QUESTION}]})
single_seconds = time.perf_counter() - start

single_text = single_result["messages"][-1].text
display(Markdown(single_text))

1. FALSO: Natal foi fundada em 25 de dezembro de 1599. ([pt.wikipedia.org](https://pt.wikipedia.org/wiki/Ponte_Newton_Navarro?utm_source=openai))

2. FALSO: A Ponte Newton Navarro foi inaugurada em 21 de novembro de 2007. ([pt.wikipedia.org](https://pt.wikipedia.org/wiki/Ponte_Newton_Navarro?utm_source=openai))

3. FALSO: A Arena das Dunas recebeu quatro jogos da Copa do Mundo de 2014. ([pt.wikipedia.org](https://pt.wikipedia.org/wiki/Ponte_Newton_Navarro?utm_source=openai))

4. FALSO: No Censo de 2022, Natal tinha cerca de 900 mil habitantes. ([pt.wikipedia.org](https://pt.wikipedia.org/wiki/Ponte_Newton_Navarro?utm_source=openai))

5. FALSO: A UFRN foi criada em 1960. ([pt.wikipedia.org](https://pt.wikipedia.org/wiki/Ponte_Newton_Navarro?utm_source=openai))

6. FALSO: Luís da Câmara Cascudo nasceu em Natal em 1898. ([pt.wikipedia.org](https://pt.wikipedia.org/wiki/Ponte_Newton_Navarro?utm_source=openai))

7. FALSO: O Instituto Metrópole Digital da UFRN foi criado em 2010. ([pt.wikipedia.org](https://pt.wikipedia.org/wiki/Ponte_Newton_Navarro?utm_source=openai))

8. FALSO: Roosevelt e Getúlio Vargas se encontraram em Natal em janeiro de 1943. ([pt.wikipedia.org](https://pt.wikipedia.org/wiki/Ponte_Newton_Navarro?utm_source=openai)) 

Três medidas acompanham o notebook: o acerto contra o gabarito, o número de URLs distintas citadas no texto final e o custo, que é a soma dos tokens de todas as chamadas ao modelo de uma execução.

In [7]:
def count_correct(text):
    """Conta os vereditos de um parecer que batem com o gabarito."""
    correct = 0
    for number, answer in enumerate(ANSWERS, start=1):
        if f"{number}. {answer}" in text:
            correct += 1
    return correct


def count_sources(text):
    """Conta as URLs distintas citadas em um texto."""
    urls = re.findall(r"https?://[^\s)]+", text)
    return len(set(urls))


def count_tokens(result):
    """Soma os tokens de todas as chamadas ao modelo de uma execução."""
    total = 0
    for message in result["messages"]:
        if message.type == "ai":
            total += message.usage_metadata["total_tokens"]
    return total

In [8]:
answer = single_result["messages"][-1]

for block in answer.content_blocks:
    if block["type"] == "server_tool_call":
        print("busca:", block["args"]["query"])

print("acertos:", count_correct(single_text))
print("fontes:", count_sources(single_text))
print("tokens:", count_tokens(single_result))

busca: fundação de Natal RN
acertos: 4
fontes: 1
tokens: 8555


O agente fez uma única busca, sobre a primeira afirmação, e citou a mesma página em todas as linhas. Ele marcou as oito afirmações como FALSO, e os dados que a busca não trouxe saíram da memória do modelo, com erros como a população de 900 mil no Censo e a criação do IMD em 2010.

## Especialização e isolamento de contexto

Um **sistema multiagente** reparte a tarefa entre agentes. Na **especialização**, cada agente recebe um papel estreito, com prompt e ferramentas próprios. No **isolamento de contexto**, cada agente trabalha em uma conversa separada e entrega só o resultado. O checador abaixo cuida de uma afirmação e devolve uma nota com a fonte; o redator não tem ferramentas e escreve a partir das notas.

In [9]:
CHECKER = """# Papel
Você é um checador de fatos.

# Tarefa
Pesquise na web a afirmação recebida e verifique se ela é verdadeira.

# Formato da resposta
A primeira linha traz só o veredito: VERDADEIRO ou FALSO.
Depois, uma nota curta com o dado correto e a URL da fonte."""

checker = create_agent(model, tools=[web_search], system_prompt=CHECKER)

In [10]:
WRITER = """# Papel
Você é um redator.

# Regras
Monte o texto pedido usando apenas as notas recebidas, com as URLs que estão nelas.
Se uma informação não estiver nas notas, escreva "não encontrado"."""

writer = create_agent(model, system_prompt=WRITER)

In [11]:
start = time.perf_counter()
results = []
notes = []

for claim in CLAIMS:
    result = checker.invoke({"messages": [{"role": "user", "content": claim}]})
    results.append(result)
    notes.append(result["messages"][-1].text)
    print(count_tokens(result), claim)

sequential_seconds = time.perf_counter() - start

display(Markdown(notes[1]))

8254 Natal foi fundada em 25 de dezembro de 1599.


8252 A Ponte Newton Navarro foi inaugurada em 2010.


8396 A Arena das Dunas recebeu seis jogos da Copa do Mundo de 2014.


8251 No Censo de 2022, Natal tinha cerca de 751 mil habitantes.


8268 A UFRN foi criada em 1958.


8252 Luís da Câmara Cascudo nasceu em Natal em 1908.


8280 O Instituto Metrópole Digital da UFRN foi criado em 2011.


8324 Roosevelt e Getúlio Vargas se encontraram em Natal em janeiro de 1944.


FALSO

A Ponte Newton Navarro foi inaugurada em 21 de novembro de 2007. ([tribunadonorte.com.br](https://tribunadonorte.com.br/natal/inaugurada-nesta-terca-a-ponte-newton-navarro/?utm_source=openai)) 

In [12]:
prompt = "# Pedido\n" + QUESTION + "\n# Notas\n" + "\n\n".join(notes)

start = time.perf_counter()
report = writer.invoke({"messages": [{"role": "user", "content": prompt}]})
writer_seconds = time.perf_counter() - start

report_text = report["messages"][-1].text
display(Markdown(report_text))
print("tokens do redator:", count_tokens(report))

1. VERDADEIRO - Natal foi fundada em 25 de dezembro de 1599. (https://natal.rn.gov.br/cidade-de-natal-rn?utm_source=openai)  
2. FALSO - A Ponte Newton Navarro foi inaugurada em 21 de novembro de 2007. (https://tribunadonorte.com.br/natal/inaugurada-nesta-terca-a-ponte-newton-navarro/?utm_source=openai)  
3. FALSO - A Arena das Dunas recebeu quatro jogos da Copa do Mundo de 2014. (https://ge.globo.com/futebol/copa-do-mundo/historia/arena-das-dunas-natal-rn.html?utm_source=openai)  
4. VERDADEIRO - No Censo de 2022, Natal tinha 751.300 habitantes. (https://www.ibge.gov.br/en/cities-and-states/rn/natal.html?utm_source=openai)  
5. VERDADEIRO - A UFRN foi criada em 25 de junho de 1958 e federalizada em 18 de dezembro de 1960. (https://www.ufrn.br/institucional/sobre-a-ufrn?utm_source=openai)  
6. FALSO - Luís da Câmara Cascudo nasceu em Natal em 30 de dezembro de 1898. (https://www.cascudo.org.br/camara-cascudo/?utm_source=openai)  
7. VERDADEIRO - O Instituto Metrópole Digital da UFRN foi criado em 18 de abril de 2011. (https://portal.imd.ufrn.br/portal/institucional/historico?utm_source=openai)  
8. FALSO - Roosevelt e Getúlio Vargas se encontraram em Natal em 28 de janeiro de 1943, não em janeiro de 1944. (https://pt.wikipedia.org/wiki/Confer%C3%AAncia_do_Potengi?utm_source=openai)

tokens do redator: 1395


Cada checador fez a própria busca e gastou cerca de 8 mil tokens só com a sua afirmação, e o parecer traz uma fonte própria em cada linha. O redator gastou pouco mais de mil tokens, porque recebeu só as notas e nenhuma linha dos resultados de busca. A coordenação é código comum: nenhum modelo decidiu quantos checadores existem nem o que cada um checa.

### Exercício 1

Peça ao redator, com as mesmas `notes`, o mesmo parecer com um dado a mais em cada linha: a data de publicação da página usada como fonte. O que o redator escreve nesse campo, e qual agente da equipe teria de mudar para o pedido ser atendido?

In [13]:
# Seu código aqui

## Paralelismo

As oito checagens não dependem uma da outra. No **paralelismo**, agentes independentes rodam ao mesmo tempo. O método `batch` roda o mesmo agente sobre várias entradas em threads paralelas, e o tempo das checagens passa a ser o da mais lenta.

In [14]:
inputs = []
for claim in CLAIMS:
    inputs.append({"messages": [{"role": "user", "content": claim}]})

start = time.perf_counter()
parallel_results = checker.batch(inputs)
parallel_seconds = time.perf_counter() - start

print(f"em sequência: {sequential_seconds:.1f} s")
print(f"em paralelo: {parallel_seconds:.1f} s")

em sequência: 17.0 s
em paralelo: 2.8 s


## Ganho e custo

As oito checagens em paralelo levaram uma fração do tempo da versão em sequência. A tabela reúne as três execuções: as duas equipes fazem as mesmas checagens e têm os mesmos acertos, fontes e tokens, e o tempo delas inclui o redator, que roda depois porque depende de todas as notas.

In [15]:
team_tokens = count_tokens(report)
for result in results:
    team_tokens += count_tokens(result)

team_correct = count_correct(report_text)
team_sources = count_sources(report_text)

comparison = pd.DataFrame(
    {
        "acertos": [count_correct(single_text), team_correct, team_correct],
        "fontes": [count_sources(single_text), team_sources, team_sources],
        "tokens": [count_tokens(single_result), team_tokens, team_tokens],
        "segundos": [
            single_seconds,
            sequential_seconds + writer_seconds,
            parallel_seconds + writer_seconds,
        ],
    },
    index=["agente único", "equipe em sequência", "equipe em paralelo"],
)

comparison.round(1)

,acertos,fontes,tokens,segundos
agente único,4,1,8555,3.5
equipe em sequência,8,8,67672,21.7
equipe em paralelo,8,8,67672,7.5


A equipe acertou os oito vereditos, cada um com a sua fonte, e gastou cerca de oito vezes os tokens do agente único, porque cada checador traz o próprio bloco de resultados de busca. Mesmo em paralelo, ela continua mais lenta, porque acrescenta a chamada do redator depois das checagens. Os quatro acertos do agente único vêm de marcar tudo como FALSO em um post com quatro afirmações falsas.

### Exercício 2

Escreva duas afirmações sobre a sua cidade, uma verdadeira e uma falsa, e rode o checador nas duas em paralelo. O checador acertou os dois vereditos, e quantos tokens as duas checagens custaram?

In [16]:
# Seu código aqui

## Quando não usar multiagentes

A divisão compensa quando a tarefa tem partes independentes e cada parte pede o próprio contexto. A checagem de uma afirmação só não tem nenhuma das duas coisas.

In [17]:
SIMPLE = "Verifique a afirmação: " + CLAIMS[1]

start = time.perf_counter()
single_simple = single.invoke({"messages": [{"role": "user", "content": SIMPLE}]})
print(f"agente único: {time.perf_counter() - start:.1f} s")
print(single_simple["messages"][-1].text)

start = time.perf_counter()
note = checker.invoke({"messages": [{"role": "user", "content": CLAIMS[1]}]})
prompt = "# Pedido\n" + SIMPLE + "\n\n# Notas\n" + note["messages"][-1].text
team_simple = writer.invoke({"messages": [{"role": "user", "content": prompt}]})
print(f"equipe: {time.perf_counter() - start:.1f} s")
print(team_simple["messages"][-1].text)

agente único: 1.9 s
A Ponte Newton Navarro foi inaugurada em 21 de novembro de 2007. ([tribunadonorte.com.br](https://tribunadonorte.com.br/natal/inaugurada-nesta-terca-a-ponte-newton-navarro/?utm_source=openai)) Portanto, a afirmação de que a ponte foi inaugurada em 2010 está incorreta. 


equipe: 3.9 s
A afirmação de que a Ponte Newton Navarro foi inaugurada em 2010 é falsa. A Ponte Newton Navarro foi inaugurada em 21 de novembro de 2007. ([tribunadonorte.com.br](https://tribunadonorte.com.br/natal/inaugurada-nesta-terca-a-ponte-newton-navarro/?utm_source=openai))


As duas versões deram o mesmo veredito, e a equipe demorou mais, porque acrescentou uma chamada ao redator a uma checagem que um agente resolve com uma busca. Tarefas acopladas também perdem com a divisão. A célula seguinte divide a redação: um redator por nota, cada um com o pedido inteiro e só a própria nota. Antes de rodar, tente prever o que sai.

In [18]:
inputs = []
for note in notes:
    prompt = "# Pedido\n" + QUESTION + "\n# Notas\n" + note
    inputs.append({"messages": [{"role": "user", "content": prompt}]})

parts = writer.batch(inputs)

for part in parts:
    print("fontes:", count_sources(part["messages"][-1].text))

display(Markdown(parts[0]["messages"][-1].text))

fontes: 1
fontes: 1
fontes: 1
fontes: 1
fontes: 1
fontes: 1
fontes: 1
fontes: 1


1. VERDADEIRO, Natal foi fundada em 25 de dezembro de 1599. Fonte: https://natal.rn.gov.br/cidade-de-natal-rn?utm_source=openai  
2. não encontrado  
3. não encontrado  
4. não encontrado  
5. não encontrado  
6. não encontrado  
7. não encontrado  
8. não encontrado

Saíram oito pareceres em vez de um, cada um com uma fonte só: a afirmação da própria nota está checada, e as outras sete ficam sem dado. O parecer é uma tarefa acoplada, porque é um texto só e depende de todas as notas juntas, e agentes isolados não veem as notas uns dos outros.

### Exercício 3

Crie um revisor, um agente sem ferramentas que recebe as `notes` da equipe e o parecer do agente único e lista as linhas do parecer em que o veredito ou o dado diverge das notas. Quantas divergências o revisor apontou, e em quantas delas o agente único citou uma fonte?

In [19]:
# Seu código aqui